In [3]:
#http://127.0.0.1:8888/?token=641ff58c730676d8f40d01ea1af56b3dc8e8be248d9aeb4a
import os
import re
import operator
import matplotlib.pyplot as plt
import warnings
import gensim
import numpy as np
warnings.filterwarnings('ignore')  # Let's not pay heed to them right now

from gensim.models import CoherenceModel, LdaModel, LsiModel, HdpModel
from gensim.models.wrappers import LdaMallet
from gensim.corpora import Dictionary
from pprint import pprint
import json
import pyLDAvis.gensim
import datetime
from nltk.stem.lancaster import LancasterStemmer
from nltk.corpus import stopwords
from nltk.stem.porter import *
import numpy as np
import string
%matplotlib inline
import pandas as pd

In [4]:
# import file to calculate LDA
# REDDIT
filename = 'libra_messages_subreddit_libra.csv'
df = pd.read_csv(os.path.join(os.getcwd(), 'reddit_data', 'clean_data', filename))
print(len(df))
# TELEGRAM
# filename = '_Telegram_cleanedData_2019-09-01_2019-09-17.csv'
# df = pd.read_csv(os.path.join(os.getcwd(), 'telegram_data', 'clean_data', filename))
# df = df.iloc[:,1:]
# print(df.columns)

873


In [69]:
df.columns

Index(['Unnamed: 0', 'subreddit', 'body', 'created_utc', 'author'], dtype='object')

In [5]:
# Select only data desidered:
# df = df[df.author == 'mindy2000']
#df1 = df[df.subreddit == 'Futurology']
print(len(df))

873


In [6]:
def build_texts(fname):
    """
    Function to build tokenized texts from file
    fname: File to be read
    Returns: yields preprocessed line
    """
    
    for line in fname:
        yield gensim.utils.simple_preprocess(line, deacc=True, min_len=3)
            

def clean_text(text):
    try:
        stop_words = set(stopwords.words('english'))
        #porter = PorterStemmer()
        #lemman = LancasterStemmer()
        translator = str.maketrans('', '', string.punctuation)
        text = re.sub(r'http\S+', '', text)
        text = text.translate(translator)
        return ' '.join([w for w in text.split() if w not in stop_words])
        #return ' '.join([lemman.stem(porter.stem(w)) for w in text.split() if w not in stop_words])

    except:
        return ''


In [7]:
# Clean text in dataframe
cleaned_data = [clean_text(d) for d in df['body'].tolist()]
train_texts = list(build_texts(cleaned_data))
print(f'Number of messages --> {len(train_texts)}')

Number of messages --> 873


In [8]:
dictionary = Dictionary(train_texts)
corpus = [dictionary.doc2bow(text) for text in train_texts]

In [9]:
import random
random.seed(113)
ldamodel = LdaModel(corpus=corpus, num_topics=5, id2word=dictionary)

In [10]:
pyLDAvis.enable_notebook()

In [11]:
pyLDAvis.gensim.prepare(ldamodel, corpus, dictionary)

BrokenProcessPool: A task has failed to un-serialize. Please ensure that the arguments of the function are all picklable.